### Plot BLASTp scores v. composite scores
### Julian Moran
### 2026-08-28

In [1]:
import boto3
import glob
import logging
import math
import os
import requests
import s3fs
import time

import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from dotenv import load_dotenv

# Env
load_dotenv("../.env", override=True)
REPO_ROOT = os.environ["INSTALL_PATH"]
MINIO_KEY = os.environ["MINIO_KEY"]
MINIO_SECRET = os.environ["MINIO_SECRET"]

# Logging
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s:%(name)s:%(message)s"
)
logger = logging.getLogger(__name__)

In [2]:
# ============================================================
#       Args
# ============================================================

# API endpoints
ENDPOINT_UNIPROT = "https://rest.uniprot.org/uniprotkb/"
ENDPOINT_UNIPROT_SEARCH = "https://rest.uniprot.org/uniprotkb/search"
ENDPOINT_UNIPROT_MAP = "https://rest.uniprot.org/idmapping"
ENDPOINT_UNIPARC_SEARCH = "https://rest.uniprot.org/uniparc/search"

# MinIO
BUCKET = "iei-project"
PREFIX_GOLD = "03_gold/defense_finder/"
PREFIX_SILVER = "02_silver/defense_finder/"
FILE_COMPOSITE = "composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet"

# Dirs
OUT_DIR_PLOT = f"{REPO_ROOT}/vis/plots"

# Check live objects in MinIO silver
client = boto3.client(
    "s3",
    endpoint_url="http://eagle.tcag.ca:9000",
    aws_access_key_id=MINIO_SECRET,
    aws_secret_access_key=MINIO_KEY,
)
response = client.list_objects_v2(
    Bucket=BUCKET,
    Prefix=PREFIX_GOLD
)
live_objects = [
    obj["Key"]
    for obj in response.get("Contents", [])
]
live_objects

['03_gold/defense_finder/composite_score.parquet/_SUCCESS',
 '03_gold/defense_finder/composite_score.parquet/part-00000-ac5574c4-cf3c-48a9-98b6-077d4db394f9-c000.snappy.parquet',
 '03_gold/defense_finder/defense_human_domain_annotated.parquet',
 '03_gold/defense_finder/final_output_spark.parquet/_SUCCESS',
 '03_gold/defense_finder/final_output_spark.parquet/part-00000-bb778272-db05-479f-a79a-2d18e89fe363-c000.snappy.parquet',
 '03_gold/defense_finder/griid_gene_subset.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.manifest.json',
 '03_gold/defense_finder/human_bacteria_structural_analogs_enriched.parquet',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/_SUCCESS',
 '03_gold/defense_finder/human_bacteria_structural_analogs_with_scores.parquet/part-00000-275fe83c-840b-476e-bf63-c47165847863-c000.snappy.parquet']

In [3]:
# ============================================================
#       In
# ============================================================

df_comp_score = pl.read_parquet(
    f"s3://{BUCKET}/{PREFIX_GOLD}{FILE_COMPOSITE}",
    storage_options={
        "aws_endpoint_url": "http://eagle.tcag.ca:9000",
        "aws_access_key_id": MINIO_SECRET,
        "aws_secret_access_key": MINIO_KEY,
    }
)
hs_accessions = df_comp_score["human_entryId"].unique()
bact_accessions = df_comp_score["defense_uniprot_ac"].unique()
logger.info(f"n unique human accessions: {len(hs_accessions)}")
logger.info(f"n unique bacterial accessions: {len(bact_accessions)}")
df_comp_score

INFO:__main__:n unique human accessions: 15572
INFO:__main__:n unique bacterial accessions: 28711


defense_uniprot_ac,human_entryId,composite_score,foldseek_evalue,pymol_rmsd,tm_score_human,tm_score_bacteria,plddt_human,plddt_bacteria
str,str,f64,f64,f32,f32,f32,f32,f32
"""A0A5C5QGP9""","""A0A024R9P6""",0.2646,0.000087,null,null,null,null,83.480003
"""A0A2R3IRC4""","""A0A0D9SF92""",0.8315,1.2070e-7,0.93,0.8169,0.1028,83.199997,77.449997
"""A0A4D8PF33""","""A0A140VK70""",0.2883,0.003094,15.01,0.2908,0.1539,80.290001,83.050003
"""A0A7S9D461""","""A0A140VK70""",0.8986,5.6920e-19,2.18,0.535,0.7678,80.290001,89.629997
"""A0A2K9LJD6""","""A0A1B0GVC6""",0.2795,0.007862,10.88,0.2428,0.29,67.610001,91.379997
…,…,…,…,…,…,…,…,…
"""A0A1D7XM13""","""Q9H4E3""",0.6409,0.000001,4.93,0.6556,0.327,84.879997,82.360001
"""A0A7D6CQE3""","""Q9H4E3""",0.5814,0.000001,5.77,0.6733,0.3337,84.879997,75.230003
"""A0A5P3ALB7""","""Q9H4E3""",0.8713,1.2930e-15,2.68,0.739,0.4786,84.879997,89.019997


In [4]:
# ============================================================
#       UniprotKB accessions --> AA seqs
# ============================================================

def get_uniprotkb_seqs(
    accessions: pl.Series,
    uniprotkb_endpoint: str,
) -> pl.DataFrame:
    """
    Submit request for AA sequence for all accessions in `accessions`.
    """

    accessions = accessions.to_list()
    query = " OR ".join(f"accession:{acc}" for acc in accessions)

    response = requests.get(
        uniprotkb_endpoint,
        params={
            "query": query,
            "format": "fasta",
            "size": len(accessions)+400,
        },
    )
    response.raise_for_status()
    records = response.text.strip().split("\n>")
    parsed_accessions = []
    sequences = []

    logger.info(
        f"UniProtKB requested: {len(accessions)}, "
        f"UniProtKB returned: {len(records)}"
    )

    for record in records:
        record = record.lstrip(">")
        lines = record.splitlines()
        header = lines[0]
        sequence = "".join(lines[1:])

        if "|" not in header:
            logger.warning(f"Unexpected FASTA header: {header!r}")
            continue

        parsed_accessions.append(header.split("|")[1])
        sequences.append(sequence)

    return pl.DataFrame({
        "uniprot_accession": parsed_accessions,
        "sequence": sequences,
    })


def get_uniprotkb_seqs_by_batch(
    accessions: pl.Series,
    uniprotkb_endpoint: str,
    batch_size: int = 100,
    wait_time: float = 0.5,
) -> pl.DataFrame:
    """
    Break up request into batches of `batch_size` and call `get_uniprotkb_seqs()` on each batch.
    @user: do not set batch_size > API batch-size limit.
    """
    # Find sequences in UniProtKB
    uniprot_batches = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        uniprot_batches.append(
            get_uniprotkb_seqs(
                accessions=batch,
                uniprotkb_endpoint=uniprotkb_endpoint
            )
        )
        logger.info(
            f"UniProtKB: processed "
            f"{min(i + batch_size, len(accessions))} "
            f"of {len(accessions)} accessions"
        )
        time.sleep(wait_time)
    df_uniprotkb_sequences = pl.concat(uniprot_batches)
    return df_uniprotkb_sequences

hs_seqs_uniprotkb = get_uniprotkb_seqs_by_batch(
    accessions=hs_accessions,
    uniprotkb_endpoint=ENDPOINT_UNIPROT_SEARCH,
    batch_size=100,
    wait_time=0.5
)
hs_seqs_uniprotkb

INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 66
INFO:__main__:UniProtKB: processed 100 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 65
INFO:__main__:UniProtKB: processed 200 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 69
INFO:__main__:UniProtKB: processed 300 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 67
INFO:__main__:UniProtKB: processed 400 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 77
INFO:__main__:UniProtKB: processed 500 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 62
INFO:__main__:UniProtKB: processed 600 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 69
INFO:__main__:UniProtKB: processed 700 of 15572 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 60
INFO:__main__:UniProtKB: processed 800 of 15572 accessions
INFO:__main__:UniProtKB 

uniprot_accession,sequence
str,str
"""Q16637""","""MAMSSGGSGGGVPEQEDSVLFRRGTGQSDD…"
"""Q8IY37""","""MGKLRRRYNIKGRQQAGPGPSKGPPEPPPV…"
"""Q8NB16""","""MENLKHIITLGQVIHKRCEEMKYCKKQCRR…"
"""Q9NZ01""","""MKHYEVEILDAKTREKLCFLDKVEPHATIA…"
"""Q02156""","""MVVFNGLLKIKICEAVSLKPTAWSLRHAVG…"
…,…
"""M0R1G9""","""MGPLSAPPCTQHITWKGLLLTASLLNFWNL…"
"""B4DR91""","""MRDGFSSWRTLEIRRFRTTARSCLNTRRST…"
"""E5RHI1""","""MERAMEQLNRLTRSLRRARTVELPEDNETA…"


In [5]:
bact_seqs_uniprotkb = get_uniprotkb_seqs_by_batch(
    accessions=bact_accessions,
    uniprotkb_endpoint=ENDPOINT_UNIPROT_SEARCH,
    batch_size=100,
    wait_time=0.5
)
bact_seqs_uniprotkb

INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 73
INFO:__main__:UniProtKB: processed 100 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 59
INFO:__main__:UniProtKB: processed 200 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 62
INFO:__main__:UniProtKB: processed 300 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 58
INFO:__main__:UniProtKB: processed 400 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 59
INFO:__main__:UniProtKB: processed 500 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 70
INFO:__main__:UniProtKB: processed 600 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 63
INFO:__main__:UniProtKB: processed 700 of 28711 accessions
INFO:__main__:UniProtKB requested: 100, UniProtKB returned: 70
INFO:__main__:UniProtKB: processed 800 of 28711 accessions
INFO:__main__:UniProtKB 

uniprot_accession,sequence
str,str
"""A0A1D8P4V5""","""MSHQSEAILEHKFIQQLVGLNYELVKVPDG…"
"""A0A1L2ZQW7""","""MTPTTSEAQRAEVHKTIWRIANDLRGSVDG…"
"""A0A1L6J5W2""","""MTATTVHGWLEKLGYTAEPGVLHLRGDAVP…"
"""A0A372KRM4""","""MKRRSYFWAKKREDSGRLLWLSLMQHLEDT…"
"""A0A4P6MVW3""","""MADSVDETIVDTLRLEVHPSVVFKLGADLI…"
…,…
"""A0A448PEC8""","""MNEPNSAYGYLPVAVSDQATVVATYERDAE…"
"""C0QCH5""","""MTSQTPEYLQSELPAIQLFQKLGYTYLDGS…"
"""A0A7L9WR34""","""MQEIVSSRRADATPHAASLIEGLRDIGYSL…"


In [16]:
# ============================================================
#       UniprotKB --> Uniparc
# ============================================================

def map_uniprotkb_to_uniparc(
    accessions: pl.Series,
    uniprot_map_endpoint: str,
    wait_time: float = 0.5
) -> pl.DataFrame:
    """
    Map UniProtKB accessions to UniParc accessions.
    Notably, the UniProt ID mapping endpoint accepts query lengths up to 100,000.
    """
    mapping_response = requests.post(
        f"{uniprot_map_endpoint}/run",
        data={
            "from": "UniProtKB_AC-ID",
            "to": "UniParc",
            "ids": ",".join(accessions.to_list()),
        },
    )
    mapping_response.raise_for_status()
    mapping_job_id = mapping_response.json()["jobId"]
    logger.info(f"Submitted request to {uniprot_map_endpoint}.")
    logger.info("Monitoring for status ...")

    while True:
        status_response = requests.get(
            f"{uniprot_map_endpoint}/status/{mapping_job_id}"
        )
        if not status_response.ok:
            raise RuntimeError(
                f"UniProt ID mapping status request failed: "
                f"{status_response.status_code} {status_response.text}"
            )
        status = status_response.json()
        if status.get("jobStatus") == "RUNNING":
            time.sleep(wait_time)
            continue
        if status.get("jobStatus") == "FAILED":
            raise RuntimeError(
                f"UniProt ID mapping failed: {status}"
            )
        break
    logger.info("Retrieving mapping results ...")

    mapping = []
    url = f"{uniprot_map_endpoint}/uniparc/results/{mapping_job_id}"
    params = {
        "format": "tsv",
        "size": 500,
    }
    while url:
        response = requests.get(url, params=params)
        response.raise_for_status()
        lines = response.text.strip().splitlines()
        if len(lines) > 1:
            mapping.extend(
                line.split("\t")[:2]
                for line in lines[1:]
            )
        url = response.links.get("next", {}).get("url")
        params = None

    df_mapping = pl.DataFrame(
        mapping,
        schema=["uniprot_accession", "uniparc_accession"],
        orient="row",
    )
    logger.info(f"UniProtKB accessions submitted: {len(accessions)}")
    logger.info(f"UniProtKB-to-UniParc mappings returned: {len(df_mapping)}")
    return df_mapping

# Get leftover UniprotKB accessions that still don't have AA seqs
hs_accessions_missing = hs_accessions.filter(
    ~hs_accessions.is_in(hs_seqs_uniprotkb["uniprot_accession"].implode())
)

# Map UniprotKB --> Uniparc
hs_accessions_uniparc = map_uniprotkb_to_uniparc(
    accessions=hs_accessions_missing,
    uniprot_map_endpoint=ENDPOINT_UNIPROT_MAP,
    wait_time=0.5
)
hs_accessions_uniparc = hs_accessions_uniparc.unique("uniprot_accession")
hs_accessions_uniparc

INFO:__main__:Submitted request to https://rest.uniprot.org/idmapping.
INFO:__main__:Monitoring for status ...
INFO:__main__:Retrieving mapping results ...
INFO:__main__:UniProtKB accessions submitted: 5334
INFO:__main__:UniProtKB-to-UniParc mappings returned: 4023


uniprot_accession,uniparc_accession
str,str
"""K7P5H8""","""UPI0001C08BEB"""
"""E2GJJ8""","""UPI0001E5A268"""
"""S4T6P7""","""UPI0003845028"""
"""A0A2U7MWF1""","""UPI000D7E986A"""
"""A0A0A0MS18""","""UPI00018920D4"""
…,…
"""A0A0A7C437""","""UPI00032B6746"""
"""D4HPK4""","""UPI000008996E"""
"""A0A0E3DC71""","""UPI00061B642B"""


In [10]:
# Get leftover UniprotKB accessions that still don't have AA seqs
bact_accessions_missing = bact_accessions.filter(
    ~bact_accessions.is_in(bact_seqs_uniprotkb["uniprot_accession"].implode())
)

# Map UniprotKB --> Uniparc
bact_accessions_uniparc =  map_uniprotkb_to_uniparc(
    accessions=bact_accessions_missing,
    uniprot_map_endpoint=ENDPOINT_UNIPROT_MAP,
    wait_time=0.5
)
bact_accessions_uniparc = bact_accessions_uniparc.unique("uniprot_accession")
bact_accessions_uniparc

INFO:__main__:Submitted request to https://rest.uniprot.org/idmapping.
INFO:__main__:Monitoring for status ...
INFO:__main__:Retrieving mapping results ...
INFO:__main__:UniProtKB accessions submitted: 10599
INFO:__main__:UniProtKB-to-UniParc mappings returned: 10616


uniprot_accession,uniparc_accession
str,str
"""A0A0F7VSU7""","""UPI000493C921"""
"""A0A3Q9UIY2""","""UPI000BC30C45"""
"""A0A6N8NDI8""","""UPI0004D4E768"""
"""A0A5N0FRQ2""","""UPI0002BB14D1"""
"""A0A653AXN5""","""UPI00027375F1"""
…,…
"""A0A7H9BYW6""","""UPI0015D04D64"""
"""A0A3E4NAL9""","""UPI0002648881"""
"""D3A0H4""","""UPI000196D83B"""


In [11]:
# ============================================================
#       Uniparc --> AA seqs
# ============================================================

def get_uniparc_seqs_by_batch(
    accessions: pl.Series,
    uniparc_endpoint: str,
    batch_size: int = 100,
    wait_time: float = 0.5
) -> pl.DataFrame:

    uniparc_batches = []
    for i in range(0, len(accessions), batch_size):
        batch = accessions[i:i + batch_size]
        query = " OR ".join(f"upi:{accession}" for accession in batch)
        response = requests.get(
            uniparc_endpoint,
            params={
                "query": query,
                "format": "fasta",
                "size": batch_size,
            },
        )
        response.raise_for_status()
        records = response.text.strip().split("\n>")
        logger.info(
            f"Uniparc requested: {len(batch)}, "
            f"Uniparc returned: {len(records)}"
        )
        sequences = []

        for record in records:
            lines = record.lstrip(">").splitlines()
            header = lines[0]
            sequence = "".join(lines[1:])
            upi = header.split()[0]
            sequences.append({
                "uniparc_accession": upi,
                "sequence": sequence,
            })

        uniparc_batches.append(pl.DataFrame(sequences))
        logger.info(
            f"UniParc: processed "
            f"{min(i + batch_size, len(accessions))} "
            f"of {len(accessions)} mapped accessions"
        )

        time.sleep(wait_time)
    return pl.concat(uniparc_batches)

hs_seqs_uniparc = get_uniparc_seqs_by_batch(
    accessions=hs_accessions_uniparc["uniparc_accession"],
    uniparc_endpoint=ENDPOINT_UNIPARC_SEARCH,
    batch_size=100,
    wait_time=0.5
)
hs_seqs_uniparc

INFO:__main__:Uniparc requested: 100, Uniparc returned: 93
INFO:__main__:UniParc: processed 100 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 89
INFO:__main__:UniParc: processed 200 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 95
INFO:__main__:UniParc: processed 300 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 90
INFO:__main__:UniParc: processed 400 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 91
INFO:__main__:UniParc: processed 500 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 87
INFO:__main__:UniParc: processed 600 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 96
INFO:__main__:UniParc: processed 700 of 4000 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 88
INFO:__main__:UniParc: processed 800 of 4000 mapped accessions
INFO:__main__:Uniparc re

uniparc_accession,sequence
str,str
"""UPI0000089B67""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""UPI000008A0FF""","""SHSMRYFYTAVSRPGRGEPHFIAVGYVDDT…"
"""UPI0001F1DCEB""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""UPI0001CF2A11""","""SHSMRYFYTAMSRPGRGEPRFITVGYVDDT…"
"""UPI0001C08BF6""","""SHSMRYFYTATSRPGRGEPRFITVGYVDDT…"
…,…
"""UPI0004E4CA2C""","""MDILVPLLQLLVLLLTLPLHLMALLGCWQP…"
"""UPI00087A1265""","""SHSMRYFYTAVSRPGRGEPHFIAVGYVDDT…"
"""UPI000CA22909""","""SHSMRYFYTAMSRPGRGEPRFIAVSYVDDT…"


In [12]:
bact_seqs_uniparc = get_uniparc_seqs_by_batch(
    accessions=bact_accessions_uniparc["uniparc_accession"],
    uniparc_endpoint=ENDPOINT_UNIPARC_SEARCH,
    batch_size=100,
    wait_time=0.5
)
bact_seqs_uniparc

INFO:__main__:Uniparc requested: 100, Uniparc returned: 99
INFO:__main__:UniParc: processed 100 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 99
INFO:__main__:UniParc: processed 200 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 99
INFO:__main__:UniParc: processed 300 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 100
INFO:__main__:UniParc: processed 400 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 100
INFO:__main__:UniParc: processed 500 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 99
INFO:__main__:UniParc: processed 600 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 100
INFO:__main__:UniParc: processed 700 of 10599 mapped accessions
INFO:__main__:Uniparc requested: 100, Uniparc returned: 100
INFO:__main__:UniParc: processed 800 of 10599 mapped accessions
INFO:__main_

uniparc_accession,sequence
str,str
"""UPI0002D3623F""","""MSKTYFNQNYNTVNTQSIQVEKKSENLNVV…"
"""UPI00000BBC93""","""MMFDDKKPAYKTDLGAMYIADSLEMLESMP…"
"""UPI0000EB9A15""","""MSWHEDPIDAAQADVAEDAYEAEPVAAVAQ…"
"""UPI0001598EE9""","""MRTLELLLNRRWILKSRERELYYQVKEELS…"
"""UPI000169F187""","""MLQSQPTPKVFISYSHDSTAHKAWVLTLAT…"
…,…
"""UPI000C2D5343""","""MKLKFNPNLEYQDEAISAIVDLFEGQNSMQ…"
"""UPI000E2B76D5""","""MNTSAIFESAGLSLRQVQQDYIEATAGALT…"
"""UPI0012B0A0D9""","""MNDYKVIAESRTFIVLDQYTREWNVAENYQ…"


In [13]:
# ============================================================
#       Wrangle -- human proteins
# ============================================================

hs_seqs = pl.concat([
    hs_seqs_uniprotkb,
    (
        hs_accessions_uniparc
        .join(
            hs_seqs_uniparc.unique("uniparc_accession"),
            on="uniparc_accession",
            how="left"
        )
        .select(["uniprot_accession", "sequence"])
    )
]).drop_nulls()
hs_seqs

uniprot_accession,sequence
str,str
"""Q16637""","""MAMSSGGSGGGVPEQEDSVLFRRGTGQSDD…"
"""Q8IY37""","""MGKLRRRYNIKGRQQAGPGPSKGPPEPPPV…"
"""Q8NB16""","""MENLKHIITLGQVIHKRCEEMKYCKKQCRR…"
"""Q9NZ01""","""MKHYEVEILDAKTREKLCFLDKVEPHATIA…"
"""Q02156""","""MVVFNGLLKIKICEAVSLKPTAWSLRHAVG…"
…,…
"""J7F7B0""","""SHSMRYFYTAMSRPGRGEPRFITVGYVDDT…"
"""I6MHI2""","""SHSMRYFYTAMSRPGRGEPRFIAVGYVDDT…"
"""D3DP96""","""MKERRASQKLSSKSIMDPNQNVKCKIVVVG…"


In [14]:
# ============================================================
#       Wrangle -- bacterial proteins
# ============================================================

bact_seqs = pl.concat([
    bact_seqs_uniprotkb,
    (
        bact_accessions_uniparc
        .join(
            bact_seqs_uniparc.unique("uniparc_accession"),
            on="uniparc_accession",
            how="left"
        )
        .select(["uniprot_accession", "sequence"])
    )
]).drop_nulls()
bact_seqs

uniprot_accession,sequence
str,str
"""A0A1D8P4V5""","""MSHQSEAILEHKFIQQLVGLNYELVKVPDG…"
"""A0A1L2ZQW7""","""MTPTTSEAQRAEVHKTIWRIANDLRGSVDG…"
"""A0A1L6J5W2""","""MTATTVHGWLEKLGYTAEPGVLHLRGDAVP…"
"""A0A372KRM4""","""MKRRSYFWAKKREDSGRLLWLSLMQHLEDT…"
"""A0A4P6MVW3""","""MADSVDETIVDTLRLEVHPSVVFKLGADLI…"
…,…
"""A0A7H9BYW6""","""MQSSSWPTVSSRALYLRAAEDLRQNQGQWD…"
"""A0A3E4NAL9""","""MSIQSEAALEAGLIATLRQMDYEYVQITEE…"
"""D3A0H4""","""MLMLITYDISLEDAEGQARLRRVAKLCLDY…"


In [ ]:
# ============================================================
#       Out
# ============================================================

hs_seqs.write_csv(
    f"{REPO_ROOT}/results/iei_human_protein_AAs.tsv",
    separator="\t"
)
bact_seqs.write_csv(
    f"{REPO_ROOT}/results/iei_bact_protein_AAs.tsv",
    separator="\t"
)